# FraudSentinel — Phase 4: Deep Learning with TensorFlow/Keras

## 1. Overview & Objective
In this phase, we design and train a **Deep Multi-Layer Perceptron (MLP)** using **TensorFlow/Keras** for binary transaction fraud detection on the preprocessed FraudSentinel datasets.

### Key Technical Requirements:
- **Architecture**: `Input(44) → Dense(128, ReLU) → Dropout(0.2) → Dense(64, ReLU) → Dropout(0.2) → Dense(32, ReLU) → Dense(1, Sigmoid)`
- **Feature Set**: 44 engineered ML features from the leakage-safe pipeline (excluding identifiers and target).
- **Imbalance Strategy**: Cost-sensitive balanced class weighting (~1.5% fraud prevalence).
- **Optimization**: Adam optimizer with `BinaryCrossentropy` loss.
- **Regularization & Callbacks**: `EarlyStopping` monitoring validation PR-AUC (`val_pr_auc`) with weight restoration.
- **Evaluation Protocol**: Threshold sensitivity analysis on validation data (0.01–0.99), followed by a single-pass evaluation on the internal test set.

In [ ]:
import os
import json
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import joblib

import tensorflow as tf
from tensorflow.keras import layers, models, callbacks, optimizers, metrics
from sklearn.preprocessing import StandardScaler
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import (
    confusion_matrix, precision_score, recall_score, f1_score,
    roc_auc_score, precision_recall_curve, auc, classification_report
)

# Set seeds for reproducibility
np.random.seed(42)
tf.random.set_seed(42)
print("TensorFlow version:", tf.__version__)
print("Num GPUs Available:", len(tf.config.list_physical_devices('GPU')))

### Markdown Analysis: Environment & Dependencies
TensorFlow and scientific libraries are initialized. Random seeds across NumPy and TensorFlow are locked to `42` to guarantee reproducible initialization of neural network weights, dropout masks, and batch shuffling.

In [ ]:
# Load processed datasets from chronological splits
train_df = pd.read_csv('../data/processed/fraudsentinel_train.csv')
val_df = pd.read_csv('../data/processed/fraudsentinel_validation.csv')
test_df = pd.read_csv('../data/processed/fraudsentinel_internal_test.csv')
blind_df = pd.read_csv('../data/processed/fraudsentinel_blind_test.csv')

print(f"Train set:         {train_df.shape[0]:,} rows, {train_df['is_fraud'].sum()} fraud ({train_df['is_fraud'].mean()*100:.2f}%)")
print(f"Validation set:    {val_df.shape[0]:,} rows, {val_df['is_fraud'].sum()} fraud ({val_df['is_fraud'].mean()*100:.2f}%)")
print(f"Internal Test set: {test_df.shape[0]:,} rows, {test_df['is_fraud'].sum()} fraud ({test_df['is_fraud'].mean()*100:.2f}%)")
print(f"Blind Test set:    {blind_df.shape[0]:,} rows (unlabeled)")

### Markdown Analysis: Dataset Partitioning & Class Imbalance
The datasets preserve the chronological sequence of banking transactions. The fraud prevalence is approximately **1.5%**, reflecting real-world banking imbalance where legitimate transactions vastly outnumber fraudulent attacks. Notice that each split maintains consistent fraud proportion without synthetic oversampling contamination.

In [ ]:
# Exclude non-numeric identifiers and target
exclude_cols = ['transaction_id', 'customer_id', 'transaction_time', 'parsed_time', 'is_fraud']
feature_cols = [c for c in train_df.columns if c not in exclude_cols]
print(f"Total input features for MLP: {len(feature_cols)}")

X_train = train_df[feature_cols].copy()
y_train = train_df['is_fraud'].values.astype(int)

X_val = val_df[feature_cols].copy()
y_val = val_df['is_fraud'].values.astype(int)

X_test = test_df[feature_cols].copy()
y_test = test_df['is_fraud'].values.astype(int)

# Fit StandardScaler strictly on the training set
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled = scaler.transform(X_val)
X_test_scaled = scaler.transform(X_test)

# Compute cost-sensitive balanced class weights
classes = np.unique(y_train)
weights = compute_class_weight('balanced', classes=classes, y=y_train)
class_weights = {int(c): float(w) for c, w in zip(classes, weights)}
print("Class weights:", class_weights)

### Markdown Analysis: Data Scaling & Leakage Prevention
- The `StandardScaler` is fitted **exclusively** on `X_train` and then applied to transform validation and test folds. This eliminates data leakage.
- Balanced class weights assign higher loss penalties to misclassified fraud examples, compensating for the severe 98.5:1.5 imbalance without perturbing the true feature distribution.

In [ ]:
# Build TensorFlow/Keras MLP Model
input_dim = len(feature_cols)

model = models.Sequential([
    layers.Input(shape=(input_dim,), name='input_features'),
    layers.Dense(128, activation='relu', name='dense_128'),
    layers.Dropout(0.2, name='dropout_1'),
    layers.Dense(64, activation='relu', name='dense_64'),
    layers.Dropout(0.2, name='dropout_2'),
    layers.Dense(32, activation='relu', name='dense_32'),
    layers.Dense(1, activation='sigmoid', name='output_fraud_prob')
], name='FraudSentinel_MLP')

model.compile(
    optimizer=optimizers.Adam(learning_rate=0.001),
    loss='binary_crossentropy',
    metrics=[
        metrics.BinaryAccuracy(name='accuracy'),
        metrics.AUC(curve='PR', name='pr_auc'),
        metrics.AUC(curve='ROC', name='roc_auc'),
        metrics.Precision(name='precision'),
        metrics.Recall(name='recall')
    ]
)
model.summary()

### Markdown Analysis: Neural Network Architecture
The network consists of three dense hidden layers progressively reducing dimensional representation (`128 → 64 → 32`) before mapping to a single sigmoid output unit. Inverted dropout rate of `0.2` prevents co-adaptation among non-linear activations.

In [ ]:
# Setup EarlyStopping on validation PR-AUC
early_stop = callbacks.EarlyStopping(
    monitor='val_pr_auc',
    mode='max',
    patience=15,
    restore_best_weights=True,
    verbose=1
)

reduce_lr = callbacks.ReduceLROnPlateau(
    monitor='val_pr_auc',
    mode='max',
    factor=0.5,
    patience=5,
    min_lr=1e-5,
    verbose=1
)

# Fit model
history = model.fit(
    X_train_scaled,
    y_train,
    validation_data=(X_val_scaled, y_val),
    epochs=100,
    batch_size=256,
    class_weight=class_weights,
    callbacks=[early_stop, reduce_lr],
    verbose=1
)

### Markdown Analysis: Training Dynamics & Early Stopping
By monitoring validation PR-AUC rather than generic cross-entropy or binary accuracy, the model halts training at peak detection capacity for the minority positive class, avoiding overtraining on the dominant negative majority.

In [ ]:
# Validation Threshold Sensitivity Analysis
val_probs = model.predict(X_val_scaled).flatten()

thresholds = np.arange(0.01, 1.00, 0.01)
val_records = []
best_th = 0.5
best_f1 = -1.0

for th in thresholds:
    th_rounded = round(float(th), 2)
    preds = (val_probs >= th_rounded).astype(int)
    p = precision_score(y_val, preds, zero_division=0)
    r = recall_score(y_val, preds, zero_division=0)
    f = f1_score(y_val, preds, zero_division=0)
    val_records.append({'threshold': th_rounded, 'precision': p, 'recall': r, 'f1': f})
    if f > best_f1:
        best_f1 = f
        best_th = th_rounded

val_grid_df = pd.DataFrame(val_records)
print(f"Best operating threshold selected: {best_th:.2f} (F1: {best_f1:.4f})")

# Plot Precision-Recall Tradeoff across thresholds
plt.figure(figsize=(10, 5))
plt.plot(val_grid_df['threshold'], val_grid_df['precision'], label='Precision', color='#3b82f6')
plt.plot(val_grid_df['threshold'], val_grid_df['recall'], label='Recall', color='#10b981')
plt.plot(val_grid_df['threshold'], val_grid_df['f1'], label='F1-Score', color='#f59e0b', linestyle='--')
plt.axvline(best_th, color='#ef4444', linestyle=':', label=f'Chosen Th={best_th:.2f}')
plt.title('Validation Threshold Sensitivity (0.01 - 0.99)', fontsize=12, fontweight='bold')
plt.xlabel('Decision Threshold')
plt.ylabel('Score')
plt.grid(True, alpha=0.3)
plt.legend()
plt.show()

### Markdown Analysis: Operating Threshold Selection
Standard `0.50` decision threshold is suboptimal under severe class imbalance. Tuning the decision threshold on validation data pinpoints the operational threshold that maximizes F1 while preserving high fraud capture recall.

In [ ]:
# Final Single-Pass Evaluation on Internal Test Set
test_probs = model.predict(X_test_scaled).flatten()
test_preds = (test_probs >= best_th).astype(int)

tn, fp, fn, tp = confusion_matrix(y_test, test_preds, labels=[0, 1]).ravel()
fpr = float(fp / (fp + tn))
fnr = float(fn / (fn + tp))
acc = float((tp + tn) / len(y_test))
prec = float(precision_score(y_test, test_preds, zero_division=0))
rec = float(recall_score(y_test, test_preds, zero_division=0))
f1 = float(f1_score(y_test, test_preds, zero_division=0))
roc_auc = float(roc_auc_score(y_test, test_probs))
precisions, recalls, _ = precision_recall_curve(y_test, test_probs)
pr_auc = float(auc(recalls, precisions))

print("==================================================")
print(f" Internal Test Metrics at Operating Threshold: {best_th:.2f}")
print("==================================================")
print(f"Accuracy:               {acc:.4f}")
print(f"Precision:              {prec:.4f}")
print(f"Recall:                 {rec:.4f}")
print(f"F1-Score:               {f1:.4f}")
print(f"ROC-AUC:                {roc_auc:.4f}")
print(f"PR-AUC:                 {pr_auc:.4f}")
print(f"False Positive Rate:    {fpr:.4f}")
print(f"False Negative Rate:    {fnr:.4f}")
print(f"Confusion Matrix:       TN={tn}, FP={fp}, FN={fn}, TP={tp}")

### Markdown Analysis: Internal Test Generalization
Evaluating the internal test set strictly once confirms that the model generalizes to unseen chronological transactions. Key business trade-offs between precision and recall are quantified through the confusion matrix.

In [ ]:
# Model & Artifact Persistence
os.makedirs('../models/deep_learning', exist_ok=True)
model.save('../models/deep_learning/fraudsentinel_mlp.keras')
joblib.dump(scaler, '../models/deep_learning/mlp_scaler.joblib')

with open('../models/deep_learning/feature_order.json', 'w') as f:
    json.dump({'feature_count': len(feature_cols), 'features': feature_cols}, f, indent=2)

print("Saved model, scaler, and feature order to models/deep_learning/")

## 6. Comprehensive Summary & Multi-Model Comparison

### Head-to-Head Comparison:
- **Logistic Regression**: High interpretability, linear decision boundary.
- **Random Forest Classifier**: High non-linear partitioning, resilient to noise.
- **TensorFlow MLP**: Smooth non-linear representations, high capacity continuous risk score calibration.

All models are tracked in `models/model_registry.json` and documented in `docs/models/deep_learning_report.md`.